In [ ]:
# ----- Core Python & Visualization ----- #
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ----- Data Preprocessing ----- #
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ----- Models ----- #
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# ----- Evaluation Metrics ----- #
from sklearn.metrics import r2_score, mean_squared_error, classification_report, confusion_matrix, accuracy_score

# ----- Neural Network (Keras) ----- #
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD



In [ ]:
# loading data
df = pd.read_csv("/Users/anhuynh/Downloads/income_evaluation.csv")
df.head()

In [ ]:
# some data cleaning
df[' income'] = df[' income'].astype(str).str.strip()
df[' income'] = df[' income'].map({'>50K': 1, '<=50K': 0})

In [ ]:
# First, make sure there are no leading spaces in column names
df.columns = df.columns.str.strip()

# Now count 0s and 1s
print(df['income'].value_counts())


In [ ]:
print(df.dtypes)


In [ ]:
# ---- (1) Load raw data ---- #
df = pd.read_csv("/Users/anhuynh/Downloads/income_evaluation.csv")
df.columns = df.columns.str.strip()


# Clean 'income' column
df['income'] = df['income'].astype(str).str.strip()
df['income'] = df['income'].map({'>50K': 1, '<=50K': 0})

# Now you can split and preprocess!
X = df.drop(columns=['income'])
y = df['income']


categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include='number').columns.tolist()

# ---- (2) Train/test split BEFORE preprocessing ---- #
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---- (3) Define Preprocessor ---- #
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Define the Logistic Regression model
logreg = LogisticRegression(solver='liblinear', random_state=42)

# 2. Build the Logistic Regression pipeline
log_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # your preprocessor for scaling + encoding
    ('classifier', logreg)
])

# 3. Fit the pipeline
log_pipeline.fit(X_train, y_train)

# 4. Predict
y_pred_log = log_pipeline.predict(X_test)

# 5. Evaluate
print("Accuracy of the Logistic Regression model:", accuracy_score(y_test, y_pred_log))
print("Confusion matrix of the Logistic Regression model:\n", confusion_matrix(y_test, y_pred_log))
print("\nClassification report of the Logistic Regression model:\n", classification_report(y_test, y_pred_log))


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Define Decision Tree model
dt = DecisionTreeClassifier(max_depth=5, random_state=42)

# 2. Build Decision Tree pipeline
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', dt)
])

# 3. Fit
dt_pipeline.fit(X_train, y_train)

# 4. Predict
y_pred_dt = dt_pipeline.predict(X_test)

# 5. Evaluate
print("Accuracy of the Decision Tree model:", accuracy_score(y_test, y_pred_dt))
print("Confusion matrix of the Decision Tree model:\n", confusion_matrix(y_test, y_pred_dt))
print("\nClassification report of the Decision Tree model:\n", classification_report(y_test, y_pred_dt))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Define Random Forest model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# 2. Build Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf)
])

# 3. Fit
rf_pipeline.fit(X_train, y_train)

# 4. Predict
y_pred_rf = rf_pipeline.predict(X_test)

# 5. Evaluate
print("Accuracy of the Random Forest model:", accuracy_score(y_test, y_pred_rf))
print("Confusion matrix of the Random Forest model:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification report of the Random Forest model:\n", classification_report(y_test, y_pred_rf))


In [ ]:
# 1. Define the Gaussian Naive Bayes model
gnb = GaussianNB()

# 2. Build the GNB pipeline
log_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # same preprocessor as others
    ('classifier', gnb)
])

# 3. Fit the GNB pipeline
log_pipeline.fit(X_train, y_train)

# 4. Predict
y_pred_gnb = gnb_pipeline.predict(X_test)

# 5. Evaluate
print("Accuracy of the Gaussian Naive Bayes model:", accuracy_score(y_test, y_pred_gnb))
print("Confusion matrix of the Gaussian Naive Bayes model:\n", confusion_matrix(y_test, y_pred_gnb))
print("\nClassification report of the Gaussian Naive Bayes model:\n", classification_report(y_test, y_pred_gnb))

In [ ]:
# ---- (4) Build Model Pipelines ---- #
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.pipeline import Pipeline

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

ada_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', AdaBoostClassifier(random_state=42))
])

# ---- (5) Train Pipelines ---- #
xgb_pipeline.fit(X_train, y_train)
ada_pipeline.fit(X_train, y_train)

# ---- (6) Predict & Evaluate ---- #
from sklearn.metrics import accuracy_score, classification_report

xgb_preds = xgb_pipeline.predict(X_test)
ada_preds = ada_pipeline.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_preds))
print("Confusion matrix of the XGboost:\n", confusion_matrix(y_test, xgb_preds))
print("\nClassification report of the XGboost model:\n", classification_report(y_test, xgb_preds))



In [ ]:
print("AdaBoost Accuracy:", accuracy_score(y_test, ada_preds))
print("Confusion matrix of the XGboost:\n", confusion_matrix(y_test, ada_preds))
print("\nClassification report of the XGboost model:\n", classification_report(y_test, ada_preds))

In [ ]:
from sklearn.naive_bayes import GaussianNB

# 1. Define the Gaussian Naive Bayes model
gnb = GaussianNB()

# 2. Build the GNB pipeline
gnb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # same preprocessor as others
    ('classifier', gnb)
])

# 3. Fit the GNB pipeline
gnb_pipeline.fit(X_train, y_train)

# 4. Predict
y_pred_gnb = gnb_pipeline.predict(X_test)

# 5. Evaluate
print("Accuracy of the Gaussian Naive Bayes model:", accuracy_score(y_test, y_pred_gnb))
print("Confusion matrix of the Gaussian Naive Bayes model:\n", confusion_matrix(y_test, y_pred_gnb))
print("\nClassification report of the Gaussian Naive Bayes model:\n", classification_report(y_test, y_pred_gnb))

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# 1. Define the MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)

# 2. Build the pipeline
mlp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # reuse the same preprocessor
    ('classifier', mlp)
])

# 3. Fit the pipeline
mlp_pipeline.fit(X_train, y_train)

# 4. Predict
y_pred_mlp = mlp_pipeline.predict(X_test)

# 5. Evaluate
print("Accuracy of the Neural Network (MLP) model:", accuracy_score(y_test, y_pred_mlp))
print("Confusion matrix of the Neural Network (MLP) model:\n", confusion_matrix(y_test, y_pred_mlp))
print("\nClassification report of the Neural Network (MLP) model:\n", classification_report(y_test, y_pred_mlp))


My XGBoost model demonstrated the strongest performance among all models tested. It achieved a high overall accuracy of 86% in predicting whether an individual earns more or less than $50,000 per year. The model was particularly effective at correctly identifying individuals earning less than $50,000, while also maintaining solid performance in recognizing higher earners. Overall, these results indicate that the model is both reliable and well-balanced, making it the most suitable option for accurately assessing income levels based on the available data